## 层归一化

由于梯度消失或梯度爆炸等问题，训练深层神经网络有时会变得具有挑战性。这些问题会导致训练过程不稳定，使网络难以有效地调整权重，从而使学习过程难以找到一组最小化损失函数的参数（权重）​。

层归一化，就是为了提高神经网络训练的稳定性和效率。层归一化的主要思想是调整神经网络层的激活（输出）​，使其均值为0且方差（单位方差）为1。这种调整有助于加速权重的有效收敛，并确保训练过程的一致性和可靠性。


In [1]:
import torch
import torch.nn as nn
torch.manual_seed(123)
batch_example=torch.randn(2,5)
layer=nn.Sequential(nn.Linear(5,6),nn.ReLU())
out=layer(batch_example)
print("out shape:", out.shape)
print(out)

out shape: torch.Size([2, 6])
tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)


为了说明问题  我们先进行一些样例实验，如上我们首先实现了一个具有5个输入和6个输出的神经网络层，并将其应用于两个输入示例，我们编写的神经网络层包括一个线性层和一个非线性激活函数ReLU（修正线性单元）​，ReLU是神经网络中的一种标准激活函数

In [2]:
mean=out.mean(dim=-1, keepdim=True)
var=out.var(dim=-1, keepdim=True)
print("mean shape:", mean.shape)
print(mean)
print(var)

mean shape: torch.Size([2, 1])
tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)
tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)


在对这些输出应用层归一化之前，先检查一下均值和方差,在这里的均值张量中，第1行是第一个输入行的均值，第2行是第二个输入行的均值,在进行均值或方差等运算时，使用keepdim=True可以确保输出张量与输入张量具有相同的维度，如果没有keepdim=True，那么返回的均值张量将是一个二维向量[0.1324, 0.2170]，而不是2×1维的矩阵[​[0.1324],[0.2170]​]。

接下来，对之前得到的层输出进行层归一化操作。具体方法是减去均值，并将结果除以方差的平方根

In [3]:
out_norm=(out-mean)/torch.sqrt(var)
mean=out_norm.mean(dim=-1, keepdim=True)
var=out_norm.var(dim=-1, keepdim=True)
print("out_norm shape:", out_norm.shape)
print(mean)
print(var)

out_norm shape: torch.Size([2, 6])
tensor([[9.9341e-09],
        [1.9868e-08]], grad_fn=<MeanBackward1>)
tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


接下来我们将把这一过程封装成一个PyTorch模块，以便在后续的GPT模型中使用

In [4]:
import torch
import torch.nn as nn
class LayerNorm(nn.Module):
    def __init__(self, emb_dim) -> None:
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))  
        self.shift = nn.Parameter(torch.zeros(emb_dim))
    
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * x_norm + self.shift

这个层归一化的具体实现作用在输入张量x的最后一个维度上，该维度对应于嵌入维度(emb_dim)。变量eps是一个小常数(epsilon)，在归一化过程中会被加到方差上以防止除零错误。

scale和shift是两个可训练的参数（与输入维度相同）​，如果在训练过程中发现调整它们可以改善模型的训练任务表现，那么大语言模型会自动进行调整。这使得模型能够学习适合其数据处理的最佳缩放和偏移。

In [5]:
torch.manual_seed(123)
batch_example = torch.randn(2, 5)

ln = LayerNorm(emb_dim=5)
out_ln = ln(batch_example)

# 这里也必须是 keepdim 小写！
mean = out_ln.mean(dim=-1, keepdim=True)
var  = out_ln.var(dim=-1, keepdim=True, unbiased=False)

print("归一化后的均值：\n", mean)
print("归一化后的方差：\n", var)

归一化后的均值：
 tensor([[-2.9802e-08],
        [ 0.0000e+00]], grad_fn=<MeanBackward1>)
归一化后的方差：
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


在我们的方差计算方法中有一个实现细节，即设置unbiased=False。这意味着在方差计算中，我们会使用样本数量n作为方差公式的除数。这种方法没有使用贝塞尔修正。贝塞尔修正通常在样本方差的估计中使用n-1作为分母，以调整偏差。因此，这种方法得到的是所谓有偏方差估计。

对于嵌入维度n非常大的大语言模型（如GPT-2）​，使用n和n-1的差异在实际中几乎可以忽略。我们选择这种方法是为了确保与GPT-2模型的归一化层兼容，并且这种方法反映了TensorFlow的默认行为，因为原始GPT-2模型是用TensorFlow实现的。使用相似的设置可以确保我们的方法与第6章中加载的预训练权重兼容。

与在批次维度上进行归一化的批归一化不同，层归一化是在特征维度上进行归一化。大语言模型通常需要大量的计算资源，训练或推理时的批次大小可能会受到硬件条件或具体用例的影响。由于层归一化是对每个输入独立进行归一化，不受批次大小的限制，因此在这些场景中它提供了更多的灵活性和稳定性。这在分布式训练或在资源受限的环境中部署模型时尤为重要。

## GELU前馈神经网络
![geluForward](imgs/geluForward.png)

ReLU（右）是一个分段线性函数，当输入为正数时直接输出输入值，否则输出0。GELU（左）则是一个平滑的非线性函数，它近似ReLU，但在几乎所有负值（除了在约等于-0.75的位置外）上都有非零梯度

GELU的平滑特性可以在训练过程中带来更好的优化效果，因为它允许模型参数进行更细微的调整。相比之下，ReLU在零点处有一个尖锐的拐角​，有时会使得优化过程更加困难，特别是在深度或复杂的网络结构中。此外，ReLU对负输入的输出为0，而GELU对负输入会输出一个小的非零值。这意味着在训练过程中，接收到负输入的神经元仍然可以参与学习，只是贡献程度不如正输入大。

In [6]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

接下里，我们将使用GELU函数来实现小型神经网络模块FeedForward，该模块将在大语言模型的Transformer块中使用。

In [7]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # 词汇表大小
    "context_length": 1024, # 上下文窗口长度
    "emb_dim": 768,         # 词元 embeding 维度
    "n_heads": 12,          # 注意力头数量
    "n_layers": 12,         # 网络层数
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

pip install torchinfo

In [8]:
from torchinfo import summary

class FeedForward(nn.Module):
    def __init__(self, cfg) -> None:
        super().__init__()
        self.layers=nn.Sequential(
            nn.Linear(cfg["emb_dim"],4*cfg["emb_dim"]),
            GELU(),
            nn.Linear(4*cfg["emb_dim"],cfg["emb_dim"])
        )
    
    def forward(self,x):
        return self.layers(x)

ffn=FeedForward(GPT_CONFIG_124M)
summary(ffn,input_size=(2,3,768))

Layer (type:depth-idx)                   Output Shape              Param #
FeedForward                              [2, 3, 768]               --
├─Sequential: 1-1                        [2, 3, 768]               --
│    └─Linear: 2-1                       [2, 3, 3072]              2,362,368
│    └─GELU: 2-2                         [2, 3, 3072]              --
│    └─Linear: 2-3                       [2, 3, 768]               2,360,064
Total params: 4,722,432
Trainable params: 4,722,432
Non-trainable params: 0
Total mult-adds (M): 9.44
Input size (MB): 0.02
Forward/backward pass size (MB): 0.18
Params size (MB): 18.89
Estimated Total Size (MB): 19.09

该模块的输入和输出维度保持一致，但它通过第一个线性层将嵌入维度扩展到了更高的维度。

扩展之后，应用非线性GELU激活函数，然后通过第二个线性变换将维度缩回原始大小。这种设计允许模型探索更丰富的表示空间

此外，输入维度和输出维度的一致性简化了架构，使我们在后续堆叠多个层时无须调整维度，从而增强了模型的扩展能力。

## 添加残差连接

残差连接最初用于计算机视觉中的深度网络（特别是残差网络）​，目的是缓解梯度消失问题。梯度消失问题指的是在训练过程中，梯度在反向传播时逐渐变小，导致早期网络层难以有效训练。

![shortcutForward](imgs/shortcutForward.png)

In [9]:
class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self,layer_sizes,use_shortcut) -> None:
        super().__init__()
        self.layers=nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0],layer_sizes[1]),GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1],layer_sizes[2]),GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2],layer_sizes[3]),GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3],layer_sizes[4]),GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4],layer_sizes[5]),GELU()),
        ])
        self.use_shortcut=use_shortcut
    def forward(self,x):
        for layer in self.layers:
            layer_output=layer(x)
            if self.use_shortcut and x.shape==layer_output.shape:
                x=x+layer_output
            else:
                x=layer_output
        return x

上述代码实现了一个具有5层的深度神经网络，每层由一个线性层和一个GELU激活函数组成。在前向传播过程中，我们通过各层迭代地传递输入。并且，如果self.use_shortcut属性被设置为True，那么我们就会选择性地添加图4-12中所示的快捷连接

我们先使用这段代码初始化一个没有快捷连接的神经网络。每一层将被初始化为接受包含3个输入值的样本，并返回3个输出值。最后一层会返回一个输出值

In [51]:
def print_gradients(model, x):
    output = model(x)
    target = torch.tensor([[0.]])

    loss = nn.MSELoss()
    loss = loss(output, target)
    
    loss.backward()

    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")


这段代码定义了一个损失函数，用于计算模型输出与用户指定目标（为简化处理，这里设为0）的接近程度。然后，当调用loss.backward()时，PyTorch会计算模型中每一层的损失梯度。我们通过model.named_parameters()迭代权重参数。假设某一层有一个3×3的权重参数矩阵，那么该层将有3×3的梯度值。我们打印这3×3的梯度值的平均绝对值，以得到每一层的单一梯度值，从而更容易比较层与层之间的梯度。

接下来，使用print_gradients函数，并将其应用于没有快捷连接的模型

In [52]:
layer_sizes = [3, 3, 3, 3, 3, 1]  
sample_input = torch.tensor([[1., 0., -1.]])
torch.manual_seed(123)
model_without_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=False
)


print_gradients(model_without_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.00020173584925942123
layers.1.0.weight has gradient mean of 0.00012011159560643137
layers.2.0.weight has gradient mean of 0.0007152040489017963
layers.3.0.weight has gradient mean of 0.0013988736318424344
layers.4.0.weight has gradient mean of 0.005049645435065031


print_gradients函数的输出显示，梯度在从最后一层(layers.4)到第1层(layers.0)的过程中逐渐变小，这种现象称为梯度消失问题。

现在，实例化一个包含残差连接的模型，并观察它的比较结果

In [53]:
torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=True
)
print_gradients(model_with_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.22169791162014008
layers.1.0.weight has gradient mean of 0.20694105327129364
layers.2.0.weight has gradient mean of 0.32896995544433594
layers.3.0.weight has gradient mean of 0.2665732204914093
layers.4.0.weight has gradient mean of 1.3258540630340576


最后一层(layers.4)的梯度仍然大于其他层。然而，梯度值在逐渐接近第1层(layers.0)时趋于稳定，并且没有缩小到几乎消失的程度。总之，快捷连接在解决深度神经网络中梯度消失问题的限制方面非常重要。快捷连接是诸如大语言模型等超大规模模型的核心构建块，它们将通过确保各层之间梯度的稳定流动来帮助实现更有效的训练。

## 构建transforer模块

![transformerBlock](imgs/transformerBlock.png)

一个Transformer块结合了多个组件，包括掩码多头注意力模块和我们之前实现的FeedForward模块​。

当Transformer块处理输入序列时，序列中的每个元素（如单词或子词）都被表示为一个固定大小的向量（此处为768维）​。Transformer块内的操作，包括多头注意力和前馈层，旨在以保持这些向量维度的方式来转换它们。

Transformer块的核心思想是，自注意力机制在多头注意力块中用于识别和分析输入序列中元素之间的关系。相比之下，前馈神经网络则在每个位置上对数据进行单独的修改。这种组合不仅提供了对输入更细致的理解和处理，而且提升了模型处理复杂数据模式的整体能力。

In [54]:
import sys
sys.path.append("..")  
from ch03.mutiAttention import MutilHeadAttention

class TransformerBlock(nn.Module):
    def __init__(self,cfg) -> None:
        super().__init__()
        self.att=MutilHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            head_nums=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
            )
        self.ff=FeedForward(cfg)
        self.norm1=LayerNorm(cfg["emb_dim"])
        self.norm2=LayerNorm(cfg["emb_dim"])
        self.drop_shortcut=nn.Dropout(cfg["drop_rate"])

    def forward(self,x):
        # 原始残差
        shorcut=x
        # 第一层归一化
        x=self.norm1(x)
        # 多头注意力模块
        x=self.att(x)
        # dropout
        x=self.drop_shortcut(x)
        # 添加原始残差
        x=x+shorcut

        shorcut=x
        x=self.norm2(x)
        x=self.ff(x)
        x=self.drop_shortcut(x)
        x=x+shorcut

        return x


这段代码定义了一个TransformerBlock类，该类在PyTorch中实现了一个多头注意力机制(MultiHeadAttention)和一个前馈神经网络(FeedForward)，两者都根据提供的配置字典（cfg，比如GPT_CONFIG_124M）进行配置。

层归一化(LayerNorm)应用于这两个组件之前，而dropout应用于这两个组件之后，以便对模型进行正则化并防止过拟合。这种方法也被称为前层归一化(Pre-LayerNorm)。较早的架构（如最初的Transformer模型）在自注意力和前馈神经网络之后才应用层归一化，这种方法被称为后层归一化(Post-LayerNorm)，这通常会导致较差的训练效果。

TransformerBlock类还实现了前向传播，其中每个组件后面都跟着一个残差连接，将块的输入加到其输出上。这个关键特性有助于在训练过程中使梯度在网络中流动，并改善深度模型的学习效果

In [55]:
"""
batch_size = 2        → 一次看 2 句话
context_length = 4    → 每句话最多 4 个词
vocab_size = 50257    → 词典有 50257 个词
emb_dim = 768         → 每个词用 768 维向量表示
"""

GPT_CONFIG_124M = {
    "vocab_size": 50257,    # 总词汇表大小，即一共有多少个单词
    "context_length": 1024, # 上下文窗口长度，即将vocab_size进行切分后，每组的大小
    "emb_dim": 768,         # 词元 embeding 维度，将单个词嵌入后的维度大小
    "n_heads": 12,          # 注意力头数量
    "n_layers": 12,         # 网络层数
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

block=TransformerBlock(GPT_CONFIG_124M)

# torch.manual_seed(123)
# x=torch.rand(2,4,768)
# output=block(x)
# print(x.shape)
# print(output.shape)
summary(block,input_size=(2, 4, 768))

Layer (type:depth-idx)                   Output Shape              Param #
TransformerBlock                         [2, 4, 768]               --
├─LayerNorm: 1-1                         [2, 4, 768]               1,536
├─MutilHeadAttention: 1-2                [2, 4, 768]               --
│    └─Linear: 2-1                       [2, 4, 768]               589,824
│    └─Linear: 2-2                       [2, 4, 768]               589,824
│    └─Linear: 2-3                       [2, 4, 768]               589,824
│    └─Dropout: 2-4                      [2, 12, 4, 4]             --
│    └─Linear: 2-5                       [2, 4, 768]               590,592
├─Dropout: 1-3                           [2, 4, 768]               --
├─LayerNorm: 1-4                         [2, 4, 768]               1,536
├─FeedForward: 1-5                       [2, 4, 768]               --
│    └─Sequential: 2-6                   [2, 4, 768]               --
│    │    └─Linear: 3-1                  [2, 4, 3072]      

## 实现GPT模型

下图展示了GPT模型的数据流。从底部开始，词元化文本首先被转换成词元嵌入，然后用位置嵌入进行增强。这些组合信息形成一个张量，然后通过中间所示的一系列Transformer块（每个块都包含多头注意力和前馈神经网络层，并带有dropout和层归一化功能）​，这些块相互堆叠并重复12次最终Transformer块的输出会经过最后一步的层归一化处理，然后传递到线性输出层。这个层会将Transformer的输出映射到一个高维空间（在本例中为50 257维，对应模型的词汇表大小）​，以预测序列中的下一个词元。

![gpt2Model](imgs/gpt2Model.png)

In [56]:
class GPTModel(nn.Module):
    def __init__(self,cfg) -> None:
        super().__init__()
        # 嵌入所有词
        self.tok_emb=nn.Embedding(cfg["vocab_size"],cfg["emb_dim"])
        # 添加位置嵌入
        self.pos_emb=nn.Embedding(cfg["context_length"],cfg["emb_dim"])
        self.drop_emb=nn.Dropout(cfg["drop_rate"])

        self.trf_blocks=nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm=LayerNorm(cfg["emb_dim"])

        self.out_head=nn.Linear(
            cfg["emb_dim"],cfg["vocab_size"],bias=False
        )

    def forward(self,in_idx):
        batch_size,seq_len=in_idx.shape
        # shape:batch_size,seq_len(context_length),emb_dim
        tok_embeds=self.tok_emb(in_idx)
        # 占位符向量torch.arange(context_length)包含一个从0开始递增，直至最大输入长度减1的数值序列
        # 因此这里其实是把位置编号 [0, 1, 2, ..., seq_len-1]转换成位置向量 [seq_len, emb_dim]
        pos_embeds=self.pos_emb(torch.arange(seq_len,device=in_idx.device))
        x=tok_embeds+pos_embeds
        x=self.drop_emb(x)
        x=self.trf_blocks(x)
        x=self.final_norm(x)
        """
        output shape:[bact_size,seq_len,vocab_size]
        第 1 维：batch（第几句话）
        第 2 维：seq_len（第几个词）
        第 3 维：vocab_size（词典里每个词的概率）
        """
        logits=self.out_head(x)
        return logits

GPTModel类的__init__构造函数通过Python字典cfg传递的配置来初始化词元嵌入层和位置嵌入层。这些嵌入层负责将输入的词元索引转换为稠密向量，并添加位置信息

__init__方法会创建一个TransformerBlock模块的顺序栈，其层数与cfg中指定的层数相同。Transformer块之后会应用一个LayerNorm层，将Transformer块的输出标准化，以稳定学习过程。最后，定义一个无偏置的线性输出头，将Transformer的输出投射到分词器的词汇空间，为词汇中的每个词元生成分数(logits)。

foward方法首先接收一批输入的词元索引，然后计算它们的嵌入表示，接着应用位置嵌入，将序列通过一系列Transformer块传递，并对最终输出进行归一化处理。最后一步是计算logits，这些logits代表了下一个词元的非归一化概率。我们将在下一节中把这些logits转换为词元和文本输出。

现在，使用传入了cfg参数的GPT_CONFIG_124M字典初始化参数量为1.24亿的GPT模型，并向其输入我们之前创建的批次文本数据

In [57]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch.shape)

torch.Size([2, 4])


In [58]:
torch.manual_seed(123)
model=GPTModel(GPT_CONFIG_124M)
summary(model,input_size=(2,4), dtypes=[torch.int64])

# out=model(batch)
# print("input batch:\n",batch)
# print("output shape:\n",out.shape)
# print(out)

Layer (type:depth-idx)                   Output Shape              Param #
GPTModel                                 [2, 4, 50257]             --
├─Embedding: 1-1                         [2, 4, 768]               38,597,376
├─Embedding: 1-2                         [4, 768]                  786,432
├─Dropout: 1-3                           [2, 4, 768]               --
├─Sequential: 1-4                        [2, 4, 768]               --
│    └─TransformerBlock: 2-1             [2, 4, 768]               --
│    │    └─LayerNorm: 3-1               [2, 4, 768]               1,536
│    │    └─MutilHeadAttention: 3-2      [2, 4, 768]               2,360,064
│    │    └─Dropout: 3-3                 [2, 4, 768]               --
│    │    └─LayerNorm: 3-4               [2, 4, 768]               1,536
│    │    └─FeedForward: 3-5             [2, 4, 768]               4,722,432
│    │    └─Dropout: 3-6                 [2, 4, 768]               --
│    └─TransformerBlock: 2-2             [2, 4, 768]

In [59]:
# 统计参数量
total_params=sum(p.numel() for p in model.parameters())
print(f"total number of paramters:{total_params}")

total number of paramters:163009536


## 文本生成
接下来，我们将编写代码，将GPT模型的张量输出转换成文本

如下图是大语言模型逐步生成文本的过程，每次生成一个词元。从初始输入上下文(“Hello, I am”)开始，模型在每轮迭代中预测下一个词元，并将其添加到输入上下文中以进行下一轮预测。如图所示，第一轮迭代添加了“a”​，第二轮迭代添加了“model”​，第三轮迭代添加了“ready”​，逐步形成了完整的句子

![textGen](imgs/textGen.png)

下图展示的下一词元生成过程说明了GPT模型如何在给定输入的情况下生成下一个词元。在每一步中，模型输出一个矩阵，其中的向量表示有可能的下一个词元。将与下一个词元对应的向量提取出来，并通过softmax函数转换为概率分布。在包含这些概率分数的向量中，找到最高值的索引，这个索引对应于词元ID。然后将这个词元ID解码为文本，生成序列中的下一个词元。最后，将这个词元附加到之前的输入中，形成新的输入序列，供下一次迭代使用。这个逐步的过程使得模型能够按顺序生成文本，从最初的输入上下文中构建连贯的短语和句子。

![textGenByGPT](imgs/textGenByGPT.png)

在实际操作中，我们会多次上述过程​，直到生成预定数量的词元。在代码中，可以按照下述步骤实现文本生成过程

In [60]:
# 给一段文本,让模型一个字一个字自动续写,直到生成 max_new_tokens 个字
# idx shape:[batch_size, seq_len]
def generate_text_simple(model,idx,max_new_tokens,context_length):
    for _ in range(max_new_tokens):
        # 使用滑动窗口，每次只把最近的 context_length 个词喂给模型
        idx_cond=idx[:,-context_length:]
        with torch.no_grad():
            # logits shape: [batch_size, seq_len, vocab_size]
            logits=model(idx_cond)
        # 模型会输出每一个位置的预测,但我们只需要最后一个位置的结果,因此shape变成[batch_size, vocab_size]
        logits=logits[:,-1,:]
        # 把输出值变成 0~1 的概率分布
        probas=torch.softmax(logits,dim=-1)
        # 选出概率最高的词
        idx_next=torch.argmax(probas,dim=-1,keepdim=True)
        # 将计算的下一个字符的索引添加到索引数组中，此时idx的shape为(batch,seq_len+1)
        idx=torch.cat((idx,idx_next),dim=-1)
    return idx

现在让我们尝试使用"Hello, I am"上下文作为模型输入来调用generate_text_simple函数。首先，将输入上下文编码为词元ID

In [61]:
start_context="Hello,I am"
encoded=tokenizer.encode(start_context)
print("encode",encoded)
# unsqueeze将encoded从[4]转换为[1,4]，为其增加一个batch维度,来适应模型输入shape
encoded_tensor=torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor:",encoded_tensor.shape)

encode [15496, 11, 40, 716]
encoded_tensor: torch.Size([1, 4])


接下来，将模型设置为.eval()模式，这将禁用诸如dropout等只在训练期间使用的随机组件。然后对编码后的输入张量使用generate_text_simple函数

In [62]:
model.eval()
out=generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=6,
    context_length=GPT_CONFIG_124M["context_length"]
)
print("output:",out)
print("output length:",len(out[0]))
# squeeze(0)，去掉批次维度
# tolist()，将PyTorch 张量变成普通 Python 列表
decoded_text=tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

output: tensor([[15496,    11,    40,   716, 27018,  7283, 46275, 11472, 21692, 43530]])
output length: 10
Hello,I am Feature IT snowball shocked merits neocons


In [63]:
model.eval()
out = generate_text_simple(
    model=model,
    idx=batch,
    max_new_tokens=6,
    context_length=GPT_CONFIG_124M["context_length"]
)

print("output shape:", out.shape)        # [2, seq_len+6]
print("output:", out)
print("output length:", len(out[0]))      # 每句话的长度

# 分别解码每句话
for i in range(out.shape[0]):
    decoded_text = tokenizer.decode(out[i].tolist())
    print(f"Sentence {i}: {decoded_text}")

output shape: torch.Size([2, 10])
output: tensor([[ 6109,  3626,  6100,   345, 37532, 24086, 47843, 30961, 42348, 15635],
        [ 6109,  1110,  6622,   257, 31387, 18590,   154, 36413, 47978, 38808]])
output length: 10
Sentence 0: Every effort moves you Aeiman Byeswickattributeometer
Sentence 1: Every day holds aftime intensive� REALLY tumble Nay
